# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

In [ ]:
# Clean consumer transactions

n = 20000

customer_id = rng.integers(1000, 6500, size=n)
year = rng.choice([2024, 2025], size=n)
basket_value = rng.lognormal(mean=np.log(50), sigma=0.7, size=n) 

basket_dict = {'customer_id': customer_id, 'year': year, 'basket_value': basket_value}

df = pd.DataFrame(basket_dict)

print(df.shape)
print(df['basket_value'].mean())
print(df['basket_value'].median())


(20000, 3)
64.53663654323408
50.25665478990503


In [ ]:
# The truth: the consumer average per year, recorded before contamination
true_mean = df.groupby('year')['basket_value'].mean()
true_growth = true_mean[2025]/true_mean[2024] - 1

df_clean = df.copy()

print(true_mean)
print(true_growth)

year
2024    64.530286
2025    64.542885
Name: basket_value, dtype: float64
0.00019525162076061697


In [ ]:
# Contamination 1: B2B orders

b2b24 = 15
b2b25 = 30
n_b2b = b2b24 + b2b25

b2b_value = rng.lognormal(mean=np.log(2000), sigma=0.5, size=n_b2b)
b2b_id = rng.integers(10000, 10020, n_b2b)
b2b_year = np.repeat([2024, 2025], [b2b24, b2b25])

b2b_dict = {'customer_id': b2b_id, 'year': b2b_year, 'basket_value': b2b_value, 'is_b2b': True}

df_b2b = pd.DataFrame(b2b_dict)
df['is_b2b'] = False

df = pd.concat([df, df_b2b], ignore_index=True)

print(df.shape)
print(df['is_b2b'].sum())


(20045, 4)
45


In [ ]:
# Contamination 2: what gets logged changes between year 1 and year 2

n_cancelled = 300

cancelled_id = rng.integers(1000, 6500, size=n_cancelled)
cancelled_value = rng.lognormal(mean=np.log(120), sigma=0.7, size=n_cancelled)
cancelled_year = np.full(n_cancelled, 2025)

canc_dict = {'customer_id': cancelled_id, 'year': cancelled_year, 'basket_value': cancelled_value, 'is_b2b': False, 'is_cancelled': True}

df_cancelled = pd.DataFrame(canc_dict)

df['is_cancelled'] = False
df = pd.concat([df, df_cancelled], ignore_index=True)

print(df.shape)
print(df['is_cancelled'].sum())

(20345, 5)
300


### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

In [ ]:
# The dashboard's number vs the truth

contaminated_mean = df.groupby('year')['basket_value'].mean()
contaminated_growth = contaminated_mean[2025]/contaminated_mean[2024] - 1

gap = contaminated_growth - true_growth

comps_dict = {'truth': true_mean, 'dashboard': contaminated_mean}
comps = pd.DataFrame(comps_dict)
comps.index.name = None


print(comps)
print(f"True growth: {true_growth:.2%}")
print(f"Contaminated growth: {contaminated_growth:.2%}")
print(f"The gap: {gap:.2%}")


          truth  dashboard
2024  64.530286  67.332107
2025  64.542885  72.436478
True growth: 0.02%
Contaminated growth: 7.58%
The gap: 7.56%


The dashboard reports a growth of +7.58%, whilst the true consumer average grew +0.02%. The gap is 7.56 percentage points, all of it from data that should not be in the metric.

---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

In [ ]:
# How much of the gap is the skew and the B2B tail?

no_b2b = df[df['is_b2b'] == False]

nob2b_mean = no_b2b.groupby('year')['basket_value'].mean()
nob2b_growth = nob2b_mean[2025]/nob2b_mean[2024] - 1

contribution_b2b = contaminated_growth - nob2b_growth

print(f'No B2B Growth: {nob2b_growth:.2%}')
print(f'B2B Contribution: {contribution_b2b:.2%}')


No B2B Growth: 3.25%
B2B Contribution: 4.33%


In [ ]:
# How much is the change in what was logged?
no_cancelled = df[df['is_cancelled'] == False]

nocancelled_mean = no_cancelled.groupby('year')['basket_value'].mean()
nocancelled_growth = nocancelled_mean[2025]/nocancelled_mean[2024] - 1

contribution_canc = contaminated_growth - nocancelled_growth

print(f'No cancellations growth: {nocancelled_growth:.2%}')
print(f'Cancellation Contribution: {contribution_canc:.2%}')


interaction = gap - (contribution_b2b + contribution_canc)

print(f'Interaction: {interaction:.2%}')

No cancellations growth: 4.75%
Cancellation Contribution: 2.84%
Interaction: 0.40%


| Mechanism | Contribution to the gap | How you isolated it |
|---|---|---|
| Skew and B2B tail | 4.33 pp | Drop the B2B rows (`is_b2b`), recompute the growth, and take the difference from the dashboard growth |
| Change in what was logged | 2.84 pp | Drop the cancelled rows (`is_cancelled`), recompute the growth, and take the difference from the dashboard growth |
| Interaction | 0.40 pp | What is left of the 7.56 pp gap after the two contributions above |
| **Total** | **7.56 pp** | Dashboard growth minus true growth |

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

In [ ]:
# Median, trimmed mean, and the mean after your B2B exclusion rule

median_by_year = df.groupby('year')['basket_value'].median()
median_growth = median_by_year[2025]/median_by_year[2024] - 1

trimmed_by_year = df.groupby('year')['basket_value'].agg(stats.trim_mean, 0.1)
trimmed_growth = trimmed_by_year[2025]/trimmed_by_year[2024] - 1

# rule uses the basket value only, not the is_b2b tag
b2b_threshold = 500
b2b_excluded = df[df['basket_value'] <= b2b_threshold]

excluded_by_year = b2b_excluded.groupby('year')['basket_value'].mean()
excluded_growth = excluded_by_year[2025]/excluded_by_year[2024] - 1

caught = (df['basket_value'] > b2b_threshold).groupby(df['is_b2b']).sum()

robust_dict = {'truth': true_mean, 'median': median_by_year, 'trimmed': trimmed_by_year, 'b2b excluded': excluded_by_year}
robust = pd.DataFrame(robust_dict)
robust.index.name = None

print(robust)
print(f"Median growth: {median_growth:.2%}")
print(f"Trimmed growth: {trimmed_growth:.2%}")
print(f"B2B excluded growth: {excluded_growth:.2%}")
print(f"Removed by the rule: {caught[True]} of {n_b2b} B2B orders, {caught[False]} consumer orders")

# the recommended metric: B2B rule on completed orders only
completed = b2b_excluded[b2b_excluded['is_cancelled'] == False]

recommended_by_year = completed.groupby('year')['basket_value'].mean()
recommended_growth = recommended_by_year[2025]/recommended_by_year[2024] - 1

print(recommended_by_year)
print(f"Recommended growth: {recommended_growth:.2%}")
print(f"True growth: {true_growth:.2%}")

          truth     median    trimmed  b2b excluded
2024  64.530286  50.061746  55.942733     64.285648
2025  64.542885  51.597383  57.693044     66.105629
Median growth: 3.07%
Trimmed growth: 3.13%
B2B excluded growth: 2.83%
Removed by the rule: 45 of 45 B2B orders, 14 consumer orders
year
2024    64.285648
2025    64.163056
Name: basket_value, dtype: float64
Recommended growth: -0.19%
True growth: 0.02%


| Estimator | Value (2025, growth) | The assumption that makes it right | When it would be wrong |
|---|---|---|---|
| Median | 51.60, +3.07% | The typical basket is what the client cares about, and contamination sits only in the tails | When the client needs the average, because revenue = orders × mean, not median. It also cannot fix a shift in the middle of the distribution, and the 300 cancelled orders still pull it up 3.07% |
| Trimmed mean | 57.69, +3.13% | Contamination is a small, symmetric share of each tail (under 10% here) | When contamination is lopsided (B2B is all top tail, so the trim also throws away genuine large consumer baskets) or when it changes the composition rather than the tails, as the cancellations do |
| Mean after B2B exclusion | 66.11, +2.83% | B2B orders can be told apart by a rule (here, baskets above 500, which catches 45 of 45 B2B orders and only 14 of 20,000 consumer orders) | When B2B and consumer baskets overlap in size, so the threshold drops real consumers or keeps real B2B. On its own it also leaves the cancellations in |

**Which number should the dashboard show?** Commit to one and defend it.

The mean after the B2B exclusion rule, computed on completed orders only. It reports 64.16 for 2025 and growth of -0.19%, against true growth of +0.02%, so it closes 7.37 of the 7.56 points of the gap.

The median and the trimmed mean answer a different question from the one the dashboard asks. The dashboard metric is an average basket, and the business uses it to build revenue as orders × average basket. A median of 51.60 can't be multiplied back into revenue. Neither one fixes the logging change either: both still report about 3% growth, because the 2025 cancelled orders shift the whole distribution and not just the tail. The two problems need two explicit rules, not a more robust statistic: one defines what a consumer order is (basket ≤ 500) and one defines what counts as an order (completed, not cancelled). Each rule can be written down and checked, and either can be changed if the business disagrees.

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

In [ ]:
import os
os.makedirs("src", exist_ok=True)

In [ ]:
%%writefile src/basket_metrics.py
"""Audit and robust-average tools for the basket-value metric."""

import numpy as np
import pandas as pd
from scipy import stats


def audit_report(df: pd.DataFrame, iqr_k: float = 1.5) -> pd.DataFrame:
    """Summarise data quality, one row per column of df.

    Parameters
    ----------
    df : pd.DataFrame
        The table to audit.
    iqr_k : float
        Outlier fence multiplier: a value is an outlier if it lies more than
        iqr_k * IQR below Q1 or above Q3.

    Returns
    -------
    pd.DataFrame
        Indexed by column name, with missing_pct, dtype, skew and n_outliers.
        Skew and outliers are NaN and 0 for non-numeric and boolean columns.
    """
    rows = {}
    for col in df.columns:
        s = df[col]
        is_numeric = pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)

        if is_numeric:
            q1, q3 = s.quantile([0.25, 0.75])
            iqr = q3 - q1
            n_outliers = int(((s < q1 - iqr_k*iqr) | (s > q3 + iqr_k*iqr)).sum())
            skew = float(s.skew())
        else:
            n_outliers = 0
            skew = np.nan

        rows[col] = {'missing_pct': s.isna().mean(), 'dtype': str(s.dtype), 'skew': skew, 'n_outliers': n_outliers}

    return pd.DataFrame.from_dict(rows, orient='index')


def robust_mean(x, method: str = "median", trim: float = 0.1, threshold: float | None = None) -> float:
    """A robust centre for x, ignoring missing values.

    Parameters
    ----------
    x : array-like
        The values to average.
    method : str
        "median", "trimmed" (drops the top and bottom `trim` share before
        averaging) or "exclusion" (the mean of the values at or below
        `threshold`).
    trim : float
        Share cut from each tail when method="trimmed".
    threshold : float, optional
        Cut-off above which values are excluded; required when
        method="exclusion".

    Returns
    -------
    float
        The robust centre.

    Raises
    ------
    ValueError
        If x has no non-missing values, method is unknown, or method is
        "exclusion" without a threshold.
    """
    values = np.asarray(x, dtype=float)
    values = values[~np.isnan(values)]

    if values.size == 0:
        raise ValueError("x has no non-missing values")

    if method == "median":
        return float(np.median(values))
    if method == "trimmed":
        return float(stats.trim_mean(values, trim))
    if method == "exclusion":
        if threshold is None:
            raise ValueError('method="exclusion" needs a threshold')
        return float(values[values <= threshold].mean())

    raise ValueError(f"unknown method: {method!r}")


if __name__ == "__main__":
    rng = np.random.default_rng(0)

    basket_value = rng.lognormal(mean=np.log(50), sigma=0.7, size=5000)
    b2b_value = rng.lognormal(mean=np.log(2000), sigma=0.5, size=20)
    demo = pd.DataFrame({'basket_value': np.concatenate([basket_value, b2b_value])})
    demo.loc[:9, 'basket_value'] = np.nan

    print(audit_report(demo))
    print(f"Mean: {demo['basket_value'].mean():.2f}")
    for method in ["median", "trimmed", "exclusion"]:
        print(f"{method}: {robust_mean(demo['basket_value'], method=method, threshold=500):.2f}")

Overwriting src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [ ]:
import importlib
import sys
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)
from basket_metrics import audit_report, robust_mean

test_rng = np.random.default_rng(7)

symmetric_data = test_rng.normal(loc=100, scale=15, size=10000)
skewed_data = test_rng.lognormal(mean=np.log(50), sigma=0.7, size=5000)
with_outlier = np.append(skewed_data, 1_000_000)

print("=== Test 1: symmetric data, robust centre = arithmetic mean ===")
for method in ["median", "trimmed"]:
    centre = robust_mean(symmetric_data, method=method)
    assert abs(centre - symmetric_data.mean()) < 1, f"Expected {method} within 1 of {symmetric_data.mean():.2f}, got {centre:.2f}"
    print(f"  {method}: {centre:.2f} vs mean {symmetric_data.mean():.2f} confirmed.")
print()

print("=== Test 2: one huge order moves the mean, not the robust centre ===")
mean_shift = with_outlier.mean() - skewed_data.mean()
assert mean_shift > 100, f"Expected the mean to move more than 100, got {mean_shift:.2f}"
print(f"  mean moved {mean_shift:.2f}")
for method in ["median", "trimmed"]:
    shift = robust_mean(with_outlier, method=method) - robust_mean(skewed_data, method=method)
    assert abs(shift) < 0.1, f"Expected {method} to move less than 0.1, got {shift:.4f}"
    print(f"  {method} moved {shift:.4f} confirmed.")
print()

print("=== Test 3: exclusion drops exactly the outlier ===")
excluded = robust_mean(with_outlier, method="exclusion", threshold=10_000)
assert np.isclose(excluded, skewed_data.mean()), f"Expected {skewed_data.mean():.2f}, got {excluded:.2f}"
untouched = robust_mean(skewed_data, method="exclusion", threshold=skewed_data.max())
assert np.isclose(untouched, skewed_data.mean()), f"Expected {skewed_data.mean():.2f}, got {untouched:.2f}"
print(f"  {excluded:.2f} = clean mean {skewed_data.mean():.2f} confirmed.")
print()

print("=== Test 4: audit_report ===")
test_df = pd.DataFrame({'basket_value': skewed_data, 'year': test_rng.choice([2024, 2025], size=5000)})
test_df.loc[:49, 'basket_value'] = np.nan
report = audit_report(test_df)
print(report)
assert len(report) == test_df.shape[1], f"Expected {test_df.shape[1]} rows, got {len(report)}"
assert np.isclose(report.loc['basket_value', 'missing_pct'], 50/5000), f"Expected 1% missing, got {report.loc['basket_value', 'missing_pct']:.2%}"
assert report.loc['basket_value', 'skew'] > 1, f"Expected skew > 1, got {report.loc['basket_value', 'skew']:.2f}"
assert report.loc['basket_value', 'n_outliers'] > 0, "Expected outliers in lognormal data, got 0"
print("  one row per column, 1% missing, right skew and outliers confirmed.")
print()

print("=== Test 5: parameter validation ===")
try:
    robust_mean(skewed_data, method="mode")
    print("  NOT caught: method='mode' was accepted")
except ValueError as e:
    print(f"  Caught expected error: {e}")

try:
    robust_mean(skewed_data, method="exclusion")
    print("  NOT caught: exclusion without a threshold was accepted")
except ValueError as e:
    print(f"  Caught expected error: {e}")

try:
    robust_mean([np.nan, np.nan])
    print("  NOT caught: all-missing input was accepted")
except ValueError as e:
    print(f"  Caught expected error: {e}")

=== Test 1: symmetric data, robust centre = arithmetic mean ===
  median: 99.80 vs mean 99.82 confirmed.
  trimmed: 99.81 vs mean 99.82 confirmed.

=== Test 2: one huge order moves the mean, not the robust centre ===
  mean moved 199.95
  median moved 0.0377 confirmed.


  trimmed moved 0.0168 confirmed.

=== Test 3: exclusion drops exactly the outlier ===
  63.71 = clean mean 63.71 confirmed.

=== Test 4: audit_report ===
              missing_pct    dtype      skew  n_outliers
basket_value         0.01  float64  2.786486         278
year                 0.00    int64  0.000000           0
  one row per column, 1% missing, right skew and outliers confirmed.

=== Test 5: parameter validation ===
  Caught expected error: unknown method: 'mode'
  Caught expected error: method="exclusion" needs a threshold
  Caught expected error: x has no non-missing values


---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`:** [ai-appendix.md](https://github.com/lukys8120/econ5200-ps1-measurement-audit/blob/main/ai-appendix.md)

### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

**The strongest objection, and the evidence that settles it:** the 500 threshold is arbitrary, and I picked it knowing the true answer, so the near-zero growth could be the product of a tuned cut-off rather than of the method.

In [ ]:
# The analysis that settles it
completed_all = df[df['is_cancelled'] == False]

thresholds = [300, 400, 500, 750, 1000, 1500, 2000, 5000]
sensitivity = {}

for t in thresholds:
    kept_by_year = completed_all[completed_all['basket_value'] <= t].groupby('year')['basket_value'].mean()
    sensitivity[t] = kept_by_year[2025]/kept_by_year[2024] - 1

sensitivity = pd.Series(sensitivity, name='growth')
sensitivity.index.name = 'threshold'

print(sensitivity.map('{:.2%}'.format))
print(f"True growth: {true_growth:.2%}")
print(f"Dashboard growth: {contaminated_growth:.2%}")

threshold
300      0.57%
400      0.07%
500     -0.19%
750     -0.25%
1000    -0.37%
1500     0.17%
2000     1.07%
5000     4.75%
Name: growth, dtype: str
True growth: 0.02%
Dashboard growth: 7.58%


**What you found, including if it changes your Phase 2 recommendation:**

The recommended growth does not depend on the exact threshold. With cancellations excluded, every cut-off from 400 to 1,500 gives growth between -0.37% and +0.17%, within 0.4 points of the true +0.02% and far from the dashboard's +7.58%. The result only breaks at the edges. At 300 the rule starts cutting into genuine consumer baskets (+0.57%), and from 2,000 up it lets B2B orders back in (+1.07% at 2,000, and +4.75% at 5,000, which is the same as applying no B2B rule). So the recommendation stands. The threshold should sit anywhere in the wide gap between the largest consumer baskets and the smallest B2B orders, and 500 is in the middle of that range. The Phase 2 recommendation does not change.

### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

The average consumer basket was flat at 64.2 in 2025 and 64.3 in 2024. A change of -0.2 percent whislt the dashboard shwoed a rise of 7.6%. The correction done was to reomove B2B orders that accounted for 4.3 points of the gap. There were aslo cancelled orders that logged only in 2025 and those accounted for 2.8 percentage points of the change (0.4 points come from the to together). The fix was to exclude the purchases over $500 and count the completed orders only. The confidance is high as any cutoff from 400 to 1500 gives a result between -0.4% and +0.2%. One thing that still needs confirmation is if there was no other logging change, plus a real B2B account flag from the CTM so the metric doesn't depend on a size threhold only.

**Average basket is flat, not up 7.6%**

- **Headline:** the average consumer basket was 64.2 in 2025 against 64.3 in 2024, a change of -0.2%. The dashboard showed +7.6%.
- **Correction:** two things inflated 2025. Large B2B orders made up 4.3 points of the gap, and cancelled orders, which were logged in 2025 but not in 2024, made up 2.8 points (0.4 points comes from the two together). The metric now excludes baskets over 500 and counts completed orders only.
- **Confidence:** high. Any cut-off between 400 and 1,500 gives a result between -0.4% and +0.2%.
- **Still needed before betting on it:** confirmation from the data team that no other logging change happened between years, and a B2B account flag from the CRM so we don't rely on a size threshold.

---
### Before you submit
- [ ] Truth recorded before contamination, and both mechanisms reported as numbers
- [ ] `src/basket_metrics.py` pushed with the notebook, and at least three assertions passing
- [ ] `ai-appendix.md` in the repository and linked above
- [ ] *Runtime › Restart and run all* runs without errors

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

In [2]:
# Clean consumer transactions

n = 20000

customer_id = rng.integers(1000, 6500, size=n)
year = rng.choice([2024, 2025], size=n)
basket_value = rng.lognormal(mean=np.log(50), sigma=0.7, size=n) 

basket_dict = {'customer_id': customer_id, 'year': year, 'basket_value': basket_value}

df = pd.DataFrame(basket_dict)

print(df.shape)
print(df['basket_value'].mean())
print(df['basket_value'].median())


(20000, 3)
64.53663654323408
50.25665478990503


In [3]:
# The truth: the consumer average per year, recorded before contamination
true_mean = df.groupby('year')['basket_value'].mean()
true_growth = true_mean[2025]/true_mean[2024] - 1

df_clean = df.copy()

print(true_mean)
print(true_growth)

year
2024    64.530286
2025    64.542885
Name: basket_value, dtype: float64
0.00019525162076061697


In [4]:
# Contamination 1: B2B orders

b2b24 = 15
b2b25 = 30
n_b2b = b2b24 + b2b25

b2b_value = rng.lognormal(mean=np.log(2000), sigma=0.5, size=n_b2b)
b2b_id = rng.integers(10000, 10020, n_b2b)
b2b_year = np.repeat([2024, 2025], [b2b24, b2b25])

b2b_dict = {'customer_id': b2b_id, 'year': b2b_year, 'basket_value': b2b_value, 'is_b2b': True}

df_b2b = pd.DataFrame(b2b_dict)
df['is_b2b'] = False

df = pd.concat([df, df_b2b], ignore_index=True)

print(df.shape)
print(df['is_b2b'].sum())


(20045, 4)
45


In [5]:
# Contamination 2: what gets logged changes between year 1 and year 2

n_cancelled = 300

cancelled_id = rng.integers(1000, 6500, size=n_cancelled)
cancelled_value = rng.lognormal(mean=np.log(120), sigma=0.7, size=n_cancelled)
cancelled_year = np.full(n_cancelled, 2025)

canc_dict = {'customer_id': cancelled_id, 'year': cancelled_year, 'basket_value': cancelled_value, 'is_b2b': False, 'is_cancelled': True}

df_cancelled = pd.DataFrame(canc_dict)

df['is_cancelled'] = False
df = pd.concat([df, df_cancelled], ignore_index=True)

print(df.shape)
print(df['is_cancelled'].sum())

(20345, 5)
300


### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

In [6]:
# The dashboard's number vs the truth

contaminated_mean = df.groupby('year')['basket_value'].mean()
contaminated_growth = contaminated_mean[2025]/contaminated_mean[2024] - 1

gap = contaminated_growth - true_growth

comps_dict = {'truth': true_mean, 'dashboard': contaminated_mean}
comps = pd.DataFrame(comps_dict)
comps.index.name = None


print(comps)
print(f"True growth: {true_growth:.2%}")
print(f"Contaminated growth: {contaminated_growth:.2%}")
print(f"The gap: {gap:.2%}")


          truth  dashboard
2024  64.530286  67.332107
2025  64.542885  72.436478
True growth: 0.02%
Contaminated growth: 7.58%
The gap: 7.56%


The dashboard reports a growth of +7.58%, whilst the true consumer average grew +0.02%. The gap is 7.56 percentage points, all of it from data that should not be in the metric.

---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

In [7]:
# How much of the gap is the skew and the B2B tail?

no_b2b = df[df['is_b2b'] == False]

nob2b_mean = no_b2b.groupby('year')['basket_value'].mean()
nob2b_growth = nob2b_mean[2025]/nob2b_mean[2024] - 1

contribution_b2b = contaminated_growth - nob2b_growth

print(f'No B2B Growth: {nob2b_growth:.2%}')
print(f'B2B Contribution: {contribution_b2b:.2%}')


No B2B Growth: 3.25%
B2B Contribution: 4.33%


In [8]:
# How much is the change in what was logged?
no_cancelled = df[df['is_cancelled'] == False]

nocancelled_mean = no_cancelled.groupby('year')['basket_value'].mean()
nocancelled_growth = nocancelled_mean[2025]/nocancelled_mean[2024] - 1

contribution_canc = contaminated_growth - nocancelled_growth

print(f'No cancellations growth: {nocancelled_growth:.2%}')
print(f'Cancellation Contribution: {contribution_canc:.2%}')


interaction = gap - (contribution_b2b + contribution_canc)

print(f'Interaction: {interaction:.2%}')

No cancellations growth: 4.75%
Cancellation Contribution: 2.84%
Interaction: 0.40%


| Mechanism | Contribution to the gap | How you isolated it |
|---|---|---|
| Skew and B2B tail | 4.33 pp | Drop the B2B rows (`is_b2b`), recompute the growth, and take the difference from the dashboard growth |
| Change in what was logged | 2.84 pp | Drop the cancelled rows (`is_cancelled`), recompute the growth, and take the difference from the dashboard growth |
| Interaction | 0.40 pp | What is left of the 7.56 pp gap after the two contributions above |
| **Total** | **7.56 pp** | Dashboard growth minus true growth |

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

In [9]:
# Median, trimmed mean, and the mean after your B2B exclusion rule

median_by_year = df.groupby('year')['basket_value'].median()
median_growth = median_by_year[2025]/median_by_year[2024] - 1

trimmed_by_year = df.groupby('year')['basket_value'].agg(stats.trim_mean, 0.1)
trimmed_growth = trimmed_by_year[2025]/trimmed_by_year[2024] - 1

# rule uses the basket value only, not the is_b2b tag
b2b_threshold = 500
b2b_excluded = df[df['basket_value'] <= b2b_threshold]

excluded_by_year = b2b_excluded.groupby('year')['basket_value'].mean()
excluded_growth = excluded_by_year[2025]/excluded_by_year[2024] - 1

caught = (df['basket_value'] > b2b_threshold).groupby(df['is_b2b']).sum()

robust_dict = {'truth': true_mean, 'median': median_by_year, 'trimmed': trimmed_by_year, 'b2b excluded': excluded_by_year}
robust = pd.DataFrame(robust_dict)
robust.index.name = None

print(robust)
print(f"Median growth: {median_growth:.2%}")
print(f"Trimmed growth: {trimmed_growth:.2%}")
print(f"B2B excluded growth: {excluded_growth:.2%}")
print(f"Removed by the rule: {caught[True]} of {n_b2b} B2B orders, {caught[False]} consumer orders")

# the recommended metric: B2B rule on completed orders only
completed = b2b_excluded[b2b_excluded['is_cancelled'] == False]

recommended_by_year = completed.groupby('year')['basket_value'].mean()
recommended_growth = recommended_by_year[2025]/recommended_by_year[2024] - 1

print(recommended_by_year)
print(f"Recommended growth: {recommended_growth:.2%}")
print(f"True growth: {true_growth:.2%}")

          truth     median    trimmed  b2b excluded
2024  64.530286  50.061746  55.942733     64.285648
2025  64.542885  51.597383  57.693044     66.105629
Median growth: 3.07%
Trimmed growth: 3.13%
B2B excluded growth: 2.83%
Removed by the rule: 45 of 45 B2B orders, 14 consumer orders
year
2024    64.285648
2025    64.163056
Name: basket_value, dtype: float64
Recommended growth: -0.19%
True growth: 0.02%


| Estimator | Value (2025, growth) | The assumption that makes it right | When it would be wrong |
|---|---|---|---|
| Median | 51.60, +3.07% | The typical basket is what the client cares about, and contamination sits only in the tails | When the client needs the average, because revenue = orders × mean, not median. It also cannot fix a shift in the middle of the distribution, and the 300 cancelled orders still pull it up 3.07% |
| Trimmed mean | 57.69, +3.13% | Contamination is a small, symmetric share of each tail (under 10% here) | When contamination is lopsided (B2B is all top tail, so the trim also throws away genuine large consumer baskets) or when it changes the composition rather than the tails, as the cancellations do |
| Mean after B2B exclusion | 66.11, +2.83% | B2B orders can be told apart by a rule (here, baskets above 500, which catches 45 of 45 B2B orders and only 14 of 20,000 consumer orders) | When B2B and consumer baskets overlap in size, so the threshold drops real consumers or keeps real B2B. On its own it also leaves the cancellations in |

**Which number should the dashboard show?** Commit to one and defend it.

The mean after the B2B exclusion rule, computed on completed orders only. It reports 64.16 for 2025 and growth of -0.19%, against true growth of +0.02%, so it closes 7.37 of the 7.56 points of the gap.

The median and the trimmed mean answer a different question from the one the dashboard asks. The dashboard metric is an average basket, and the business uses it to build revenue as orders × average basket. A median of 51.60 can't be multiplied back into revenue. Neither one fixes the logging change either: both still report about 3% growth, because the 2025 cancelled orders shift the whole distribution and not just the tail. The two problems need two explicit rules, not a more robust statistic: one defines what a consumer order is (basket ≤ 500) and one defines what counts as an order (completed, not cancelled). Each rule can be written down and checked, and either can be changed if the business disagrees.

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

In [10]:
import os
os.makedirs("src", exist_ok=True)

In [11]:
%%writefile src/basket_metrics.py
"""Audit and robust-average tools for the basket-value metric."""

import numpy as np
import pandas as pd
from scipy import stats


def audit_report(df: pd.DataFrame, iqr_k: float = 1.5) -> pd.DataFrame:
    """Summarise data quality, one row per column of df.

    Parameters
    ----------
    df : pd.DataFrame
        The table to audit.
    iqr_k : float
        Outlier fence multiplier: a value is an outlier if it lies more than
        iqr_k * IQR below Q1 or above Q3.

    Returns
    -------
    pd.DataFrame
        Indexed by column name, with missing_pct, dtype, skew and n_outliers.
        Skew and outliers are NaN and 0 for non-numeric and boolean columns.
    """
    rows = {}
    for col in df.columns:
        s = df[col]
        is_numeric = pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)

        if is_numeric:
            q1, q3 = s.quantile([0.25, 0.75])
            iqr = q3 - q1
            n_outliers = int(((s < q1 - iqr_k*iqr) | (s > q3 + iqr_k*iqr)).sum())
            skew = float(s.skew())
        else:
            n_outliers = 0
            skew = np.nan

        rows[col] = {'missing_pct': s.isna().mean(), 'dtype': str(s.dtype), 'skew': skew, 'n_outliers': n_outliers}

    return pd.DataFrame.from_dict(rows, orient='index')


def robust_mean(x, method: str = "median", trim: float = 0.1, threshold: float | None = None) -> float:
    """A robust centre for x, ignoring missing values.

    Parameters
    ----------
    x : array-like
        The values to average.
    method : str
        "median", "trimmed" (drops the top and bottom `trim` share before
        averaging) or "exclusion" (the mean of the values at or below
        `threshold`).
    trim : float
        Share cut from each tail when method="trimmed".
    threshold : float, optional
        Cut-off above which values are excluded; required when
        method="exclusion".

    Returns
    -------
    float
        The robust centre.

    Raises
    ------
    ValueError
        If x has no non-missing values, method is unknown, or method is
        "exclusion" without a threshold.
    """
    values = np.asarray(x, dtype=float)
    values = values[~np.isnan(values)]

    if values.size == 0:
        raise ValueError("x has no non-missing values")

    if method == "median":
        return float(np.median(values))
    if method == "trimmed":
        return float(stats.trim_mean(values, trim))
    if method == "exclusion":
        if threshold is None:
            raise ValueError('method="exclusion" needs a threshold')
        return float(values[values <= threshold].mean())

    raise ValueError(f"unknown method: {method!r}")


if __name__ == "__main__":
    rng = np.random.default_rng(0)

    basket_value = rng.lognormal(mean=np.log(50), sigma=0.7, size=5000)
    b2b_value = rng.lognormal(mean=np.log(2000), sigma=0.5, size=20)
    demo = pd.DataFrame({'basket_value': np.concatenate([basket_value, b2b_value])})
    demo.loc[:9, 'basket_value'] = np.nan

    print(audit_report(demo))
    print(f"Mean: {demo['basket_value'].mean():.2f}")
    for method in ["median", "trimmed", "exclusion"]:
        print(f"{method}: {robust_mean(demo['basket_value'], method=method, threshold=500):.2f}")

Overwriting src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [12]:
import importlib
import sys
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)
from basket_metrics import audit_report, robust_mean

test_rng = np.random.default_rng(7)

symmetric_data = test_rng.normal(loc=100, scale=15, size=10000)
skewed_data = test_rng.lognormal(mean=np.log(50), sigma=0.7, size=5000)
with_outlier = np.append(skewed_data, 1_000_000)

print("=== Test 1: symmetric data, robust centre = arithmetic mean ===")
for method in ["median", "trimmed"]:
    centre = robust_mean(symmetric_data, method=method)
    assert abs(centre - symmetric_data.mean()) < 1, f"Expected {method} within 1 of {symmetric_data.mean():.2f}, got {centre:.2f}"
    print(f"  {method}: {centre:.2f} vs mean {symmetric_data.mean():.2f} confirmed.")
print()

print("=== Test 2: one huge order moves the mean, not the robust centre ===")
mean_shift = with_outlier.mean() - skewed_data.mean()
assert mean_shift > 100, f"Expected the mean to move more than 100, got {mean_shift:.2f}"
print(f"  mean moved {mean_shift:.2f}")
for method in ["median", "trimmed"]:
    shift = robust_mean(with_outlier, method=method) - robust_mean(skewed_data, method=method)
    assert abs(shift) < 0.1, f"Expected {method} to move less than 0.1, got {shift:.4f}"
    print(f"  {method} moved {shift:.4f} confirmed.")
print()

print("=== Test 3: exclusion drops exactly the outlier ===")
excluded = robust_mean(with_outlier, method="exclusion", threshold=10_000)
assert np.isclose(excluded, skewed_data.mean()), f"Expected {skewed_data.mean():.2f}, got {excluded:.2f}"
untouched = robust_mean(skewed_data, method="exclusion", threshold=skewed_data.max())
assert np.isclose(untouched, skewed_data.mean()), f"Expected {skewed_data.mean():.2f}, got {untouched:.2f}"
print(f"  {excluded:.2f} = clean mean {skewed_data.mean():.2f} confirmed.")
print()

print("=== Test 4: audit_report ===")
test_df = pd.DataFrame({'basket_value': skewed_data, 'year': test_rng.choice([2024, 2025], size=5000)})
test_df.loc[:49, 'basket_value'] = np.nan
report = audit_report(test_df)
print(report)
assert len(report) == test_df.shape[1], f"Expected {test_df.shape[1]} rows, got {len(report)}"
assert np.isclose(report.loc['basket_value', 'missing_pct'], 50/5000), f"Expected 1% missing, got {report.loc['basket_value', 'missing_pct']:.2%}"
assert report.loc['basket_value', 'skew'] > 1, f"Expected skew > 1, got {report.loc['basket_value', 'skew']:.2f}"
assert report.loc['basket_value', 'n_outliers'] > 0, "Expected outliers in lognormal data, got 0"
print("  one row per column, 1% missing, right skew and outliers confirmed.")
print()

print("=== Test 5: parameter validation ===")
try:
    robust_mean(skewed_data, method="mode")
    print("  NOT caught: method='mode' was accepted")
except ValueError as e:
    print(f"  Caught expected error: {e}")

try:
    robust_mean(skewed_data, method="exclusion")
    print("  NOT caught: exclusion without a threshold was accepted")
except ValueError as e:
    print(f"  Caught expected error: {e}")

try:
    robust_mean([np.nan, np.nan])
    print("  NOT caught: all-missing input was accepted")
except ValueError as e:
    print(f"  Caught expected error: {e}")

=== Test 1: symmetric data, robust centre = arithmetic mean ===
  median: 99.80 vs mean 99.82 confirmed.
  trimmed: 99.81 vs mean 99.82 confirmed.

=== Test 2: one huge order moves the mean, not the robust centre ===
  mean moved 199.95
  median moved 0.0377 confirmed.


  trimmed moved 0.0168 confirmed.

=== Test 3: exclusion drops exactly the outlier ===
  63.71 = clean mean 63.71 confirmed.

=== Test 4: audit_report ===
              missing_pct    dtype      skew  n_outliers
basket_value         0.01  float64  2.786486         278
year                 0.00    int64  0.000000           0
  one row per column, 1% missing, right skew and outliers confirmed.

=== Test 5: parameter validation ===
  Caught expected error: unknown method: 'mode'
  Caught expected error: method="exclusion" needs a threshold
  Caught expected error: x has no non-missing values


---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`:** [ai-appendix.md](https://github.com/lukys8120/econ5200-ps1-measurement-audit/blob/main/ai-appendix.md)

### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

**The strongest objection, and the evidence that settles it:** the 500 threshold is arbitrary, and I picked it knowing the true answer, so the near-zero growth could be the product of a tuned cut-off rather than of the method.

In [13]:
# The analysis that settles it
completed_all = df[df['is_cancelled'] == False]

thresholds = [300, 400, 500, 750, 1000, 1500, 2000, 5000]
sensitivity = {}

for t in thresholds:
    kept_by_year = completed_all[completed_all['basket_value'] <= t].groupby('year')['basket_value'].mean()
    sensitivity[t] = kept_by_year[2025]/kept_by_year[2024] - 1

sensitivity = pd.Series(sensitivity, name='growth')
sensitivity.index.name = 'threshold'

print(sensitivity.map('{:.2%}'.format))
print(f"True growth: {true_growth:.2%}")
print(f"Dashboard growth: {contaminated_growth:.2%}")

threshold
300      0.57%
400      0.07%
500     -0.19%
750     -0.25%
1000    -0.37%
1500     0.17%
2000     1.07%
5000     4.75%
Name: growth, dtype: str
True growth: 0.02%
Dashboard growth: 7.58%


**What you found, including if it changes your Phase 2 recommendation:**

The recommended growth does not depend on the exact threshold. With cancellations excluded, every cut-off from 400 to 1,500 gives growth between -0.37% and +0.17%, within 0.4 points of the true +0.02% and far from the dashboard's +7.58%. The result only breaks at the edges. At 300 the rule starts cutting into genuine consumer baskets (+0.57%), and from 2,000 up it lets B2B orders back in (+1.07% at 2,000, and +4.75% at 5,000, which is the same as applying no B2B rule). So the recommendation stands. The threshold should sit anywhere in the wide gap between the largest consumer baskets and the smallest B2B orders, and 500 is in the middle of that range. The Phase 2 recommendation does not change.

### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

The average consumer basket was flat at 64.2 in 2025 and 64.3 in 2024. A change of -0.2 percent whislt the dashboard shwoed a rise of 7.6%. The correction done was to reomove B2B orders that accounted for 4.3 points of the gap. There were aslo cancelled orders that logged only in 2025 and those accounted for 2.8 percentage points of the change (0.4 points come from the to together). The fix was to exclude the purchases over $500 and count the completed orders only. The confidance is high as any cutoff from 400 to 1500 gives a result between -0.4% and +0.2%. One thing that still needs confirmation is if there was no other logging change, plus a real B2B account flag from the CTM so the metric doesn't depend on a size threhold only.

**Average basket is flat, not up 7.6%**

- **Headline:** the average consumer basket was 64.2 in 2025 against 64.3 in 2024, a change of -0.2%. The dashboard showed +7.6%.
- **Correction:** two things inflated 2025. Large B2B orders made up 4.3 points of the gap, and cancelled orders, which were logged in 2025 but not in 2024, made up 2.8 points (0.4 points comes from the two together). The metric now excludes baskets over 500 and counts completed orders only.
- **Confidence:** high. Any cut-off between 400 and 1,500 gives a result between -0.4% and +0.2%.
- **Still needed before betting on it:** confirmation from the data team that no other logging change happened between years, and a B2B account flag from the CRM so we don't rely on a size threshold.

---
### Before you submit
- [ ] Truth recorded before contamination, and both mechanisms reported as numbers
- [ ] `src/basket_metrics.py` pushed with the notebook, and at least three assertions passing
- [ ] `ai-appendix.md` in the repository and linked above
- [ ] *Runtime › Restart and run all* runs without errors